<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Avalon-ST Bus Helpers

This tutorial drives, receives, and monitors packetized streaming data with cocotb. It requires a simulator DUT with Avalon-ST signals.

## 1. Signals and transfers

A beat is accepted when `valid` and the ready condition for the configured latency are both satisfied. `data` is required; `valid`, `ready`, `startofpacket`, `endofpacket`, `empty`, `error`, and `channel` are optional. `AvalonFormat` describes only the symbols packed into `data`; see `02_video_frames.ipynb` for a detailed format example.

`AvalonSTBus.from_prefix(dut, "input")` binds signals such as `dut.input_data` and `dut.input_valid`. Use the constructor directly when signal names do not share a prefix.

## 2. Source and sink loopback

`AvalonSTSource` queues frames and drives `valid`, packet boundaries, sidebands, and packed data. `AvalonSTSink` drives `ready`, reconstructs accepted beats into frames, and queues them for `recv()`.

In [ ]:
import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge

from fpga_verification.sim.buses import (
    AvalonFormat,
    AvalonSTBeat,
    AvalonSTBus,
    AvalonSTFrame,
    AvalonSTSink,
    AvalonSTSource,
)


def pause_every_fourth_cycle():
    while True:
        yield False
        yield False
        yield False
        yield True


@cocotb.test()
async def stream_loopback_test(dut):
    cocotb.start_soon(Clock(dut.clk, 10, units="ns").start())

    dut.reset.value = 1
    await RisingEdge(dut.clk)
    dut.reset.value = 0

    fmt = AvalonFormat(
        bits_per_symbol=8,
        symbols_per_beat=4,
        first_symbol_in_high_order_bits=False,
    )

    source = AvalonSTSource(
        AvalonSTBus.from_prefix(dut, "sink"),
        fmt,
        dut.clk,
        reset=dut.reset,
        packets=True,
        idle_value=0,
    )
    sink = AvalonSTSink(
        AvalonSTBus.from_prefix(dut, "source"),
        fmt,
        dut.clk,
        reset=dut.reset,
        packets=True,
    )
    sink.set_pause_generator(pause_every_fourth_cycle())

    await source.send(AvalonSTFrame([0x11, 0x22, 0x33, 0x44, 0x55]))
    received = await sink.recv()

    assert received.data == [0x11, 0x22, 0x33, 0x44, 0x55]

`AvalonSTMonitor` is passive; use it when the testbench must observe an existing valid/ready stream without driving ready.

In [ ]:
from fpga_verification.sim.buses import AvalonSTMonitor

# monitor = AvalonSTMonitor(AvalonSTBus.from_prefix(dut, "tap"), fmt, dut.clk,
#                           reset=dut.reset, packets=True)
# frame = await monitor.recv()
# beat = await monitor.recv_beat()

## 3. AvalonSTFrame and AvalonSTBeat

`AvalonSTFrame.data` is a list of unpacked symbols, not packed bus words. The source groups these symbols according to `symbols_per_beat`. `channel` and `error` may be one value for the whole frame or a list of values selected as the frame advances. `tx_complete` may be a cocotb `Event` or callback invoked after the final beat is transferred.

`AvalonSTBeat` is the exact accepted bus transfer: packed `data`, unpacked `symbols`, SOP/EOP, `empty`, `error`, `channel`, and simulation timestamp. Use `recv()` for complete packets and `recv_beat()` for cycle-level inspection.

In [ ]:
frame = AvalonSTFrame(
    data=[0x11, 0x22, 0x33, 0x44, 0x55],
    channel=2,
    error=0,
)
print(frame)

beat = AvalonSTBeat(
    data=0x44332211,
    symbols=[0x11, 0x22, 0x33, 0x44],
    sop=1,
    eop=0,
)
print(beat)

# Inside a cocotb coroutine:
# await source.send(frame)
# complete_packet = await sink.recv()
# first_transfer = await sink.recv_beat()

## 4. Packet modes and empty symbols

With `packets=True`, SOP/EOP delimit each `AvalonSTFrame`. If the final beat is incomplete, `empty` gives the number of unused symbols. With `packets=False`, every accepted beat becomes an independent frame and packet signals are ignored. The default `packets=None` enables packet mode automatically when both SOP and EOP signals exist.

For four symbols per beat, a five-symbol frame uses two beats. The second beat contains one data symbol, three zero padding symbols, and `empty=3`. The received frame contains only the original five symbols.

## 5. Backpressure and pause generators

`source.pause = True` stops the source from presenting new data. `sink.pause = True` deasserts its ready behavior. `set_pause_generator()` applies a sequence of boolean pause values on consecutive clocks, which is useful for deterministic or randomized backpressure tests. Call `clear_pause_generator()` to stop it.

Source and sink queues expose occupancy counters and optional symbol/frame limits. `send_nowait()` and `recv_nowait()` are available when coroutine blocking is undesirable. `source.wait()` waits until all queued traffic has completed.

In [ ]:
def random_pause(seed=1, probability=0.25):
    import random

    rng = random.Random(seed)
    while True:
        yield rng.random() < probability


# source.set_pause_generator(random_pause(seed=1))
# sink.set_pause_generator(random_pause(seed=2))

## 6. Ready timing, reset, and diagnostics

The helpers currently support `(ready_latency, ready_allowance)` modes `(0, 0)` and `(1, 1)`. Set `strict_ready_latency=True` on a monitor when an RL=1 protocol violation should fail immediately. `timeout_cycles` detects a stream that stops transferring.

During reset, partial receive packets are discarded and the source returns to idle. `reset_active_level` selects active-high or active-low reset. `cancel()` stops background coroutines and `clear()` removes queued data. Monitor errors include a snapshot of the relevant Avalon-ST signals to make malformed SOP/EOP, `empty`, and X/Z failures easier to diagnose.